In [ ]:
# !pip install catboost

In [6]:
# =========================
# 1. Import Libraries
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR

In [3]:
from catboost import CatBoostRegressor

In [9]:
# Took 20 seconds
# =========================
# 2. Load Data
# =========================
df = pd.read_csv('data-movies2.csv')

# =========================
# 3. Feature Engineering
# =========================

# Convert release_date → year
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year

# Drop unnecessary columns
df = df.drop(columns=['id', 'title', 'release_date'])

# =========================
# 4. Define Target
# =========================
target_col = 'vote_average'

X = df.drop(columns=[target_col])
y = df[target_col]

# =========================
# 5. Handle Missing Values
# =========================
X = X.fillna(0)

# =========================
# 6. Separate Categorical Columns
# =========================
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# One-hot encoding for sklearn models
X_encoded = pd.get_dummies(X, drop_first=True)

# =========================
# 7. Train-Test Split
# =========================
X_train_enc, X_test_enc, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

X_train_cat, X_test_cat, _, _ = train_test_split(
    X, y, test_size=0.2, random_state=42
)


results = []

# =========================
# 8. CatBoost Model
# =========================
cat_model = CatBoostRegressor(verbose=0)

cat_model.fit(
    X_train_cat, y_train,
    cat_features=cat_cols
)

cat_rmse = np.sqrt(mean_squared_error(y_test, cat_preds))
cat_r2 = r2_score(y_test, cat_preds)

results.append(("CatBoost", cat_rmse, cat_r2))

results_df = pd.DataFrame(results, columns=["Model", "RMSE", "R-square"])
results_df = results_df.sort_values(by="RMSE")

print("\nModel Performance (Lower RMSE is better):\n")
print(results_df)


Model Performance (Lower RMSE is better):

      Model      RMSE  R-square
0  CatBoost  0.553187  0.479248



Model Performance (Lower RMSE is better):

      Model      RMSE  R-square
0  CatBoost  0.553187  0.479248


# Compare with others

In [12]:
# 1) Run all other models first
import time

start = time.time()
# =========================
# Initialize Models
# =========================
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(),
    # "Gradient Boosting": GradientBoostingRegressor(),
    # "SVR": SVR()
}

# =========================
# Train & Evaluate Models
# =========================
results = []

for name, model in models.items():
    model.fit(X_train_enc, y_train)
    preds = model.predict(X_test_enc)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results.append((name, rmse, r2))


end = time.time()
print("time taken:", end-start)


time taken: 62.335907220840454


In [13]:
# 2) Now run catboost
start = time.time()
cat_model = CatBoostRegressor(verbose=0)

cat_model.fit(
    X_train_cat, y_train,
    cat_features=cat_cols
)
end = time.time()
print("time taken:", end-start)

cat_rmse = np.sqrt(mean_squared_error(y_test, cat_preds))
cat_r2 = r2_score(y_test, cat_preds)

results.append(("CatBoost", cat_rmse, cat_r2))

# =========================
# Results Comparison
# =========================
results_df = pd.DataFrame(results, columns=["Model", "RMSE", "R-square"])
results_df = results_df.sort_values(by="RMSE")

print("\nModel Performance (Lower RMSE is better):\n")
print(results_df)

time taken: 26.74623394012451

Model Performance (Lower RMSE is better):

               Model      RMSE  R-square
2           CatBoost  0.553187  0.479248
1      Random Forest  0.585727  0.416182
0  Linear Regression  0.649731  0.281618
